In [1]:
import pandas as pd
import sqlite3

con = sqlite3.connect('../data/checking-logs.sqlite')

In [2]:
schema = pd.read_sql_query("PRAGMA table_info(test);", con)

print(schema)

   cid             name       type  notnull dflt_value  pk
0    0              uid       TEXT        0       None   0
1    1          labname       TEXT        0       None   0
2    2  first_commit_ts  TIMESTAMP        0       None   0
3    3    first_view_ts  TIMESTAMP        0       None   0


In [3]:
f_10 = pd.read_sql_query('SELECT * FROM pageviews LIMIT 10',con)
print(f_10)

   index      uid                    datetime
0      0  admin_1  2020-04-17 12:01:08.463179
1      1  admin_1  2020-04-17 12:01:23.743946
2      2  admin_3  2020-04-17 12:17:39.287778
3      3  admin_3  2020-04-17 12:17:40.001768
4      4  admin_1  2020-04-17 12:27:30.646665
5      5  admin_1  2020-04-17 12:35:44.884757
6      6  admin_1  2020-04-17 12:35:52.735016
7      7  admin_3  2020-04-17 12:36:21.401412
8      8  admin_3  2020-04-17 12:36:22.023355
9      9  admin_1  2020-04-17 13:55:19.129243


In [4]:
f_10 = pd.read_sql_query('SELECT * FROM pageviews LIMIT 10',con)
print(f_10)

   index      uid                    datetime
0      0  admin_1  2020-04-17 12:01:08.463179
1      1  admin_1  2020-04-17 12:01:23.743946
2      2  admin_3  2020-04-17 12:17:39.287778
3      3  admin_3  2020-04-17 12:17:40.001768
4      4  admin_1  2020-04-17 12:27:30.646665
5      5  admin_1  2020-04-17 12:35:44.884757
6      6  admin_1  2020-04-17 12:35:52.735016
7      7  admin_3  2020-04-17 12:36:21.401412
8      8  admin_3  2020-04-17 12:36:22.023355
9      9  admin_1  2020-04-17 13:55:19.129243


In [5]:
df_min = pd.read_sql_query("""
SELECT uid, min_diff
FROM (
    SELECT
        t.uid,
        MIN(
            CAST((unixepoch(t.first_commit_ts) - d.deadlines) / 3600 AS INTEGER)
        ) AS min_diff
    FROM test t
    JOIN deadlines d
      ON t.labname = d.labs
    WHERE t.labname <> 'project1'
    GROUP BY t.uid
)
ORDER BY min_diff ASC
LIMIT 1
""", con)

df_min

,uid,min_diff
0,user_30,-202


In [ ]:
df_max = pd.read_sql_query("""
SELECT uid, max_diff
FROM (
    SELECT
        t.uid,
        MAX(
            CAST((unixepoch(t.first_commit_ts) - d.deadlines) / 3600 AS INTEGER)
        ) AS max_diff
    FROM test t
    JOIN deadlines d
      ON t.labname = d.labs
    WHERE t.labname <> 'project1'
    GROUP BY t.uid
)
ORDER BY max_diff DESC
LIMIT 1
""", con)

df_max

,uid,max_diff
0,user_25,-2


In [7]:
df_avg = pd.read_sql_query("""
SELECT
    AVG(
        CAST((unixepoch(t.first_commit_ts) - d.deadlines) / 3600 AS INTEGER)
    ) AS avg_diff
FROM test t
JOIN deadlines d
  ON t.labname = d.labs
WHERE t.labname <> 'project1'
""", con)

df_avg

,avg_diff
0,-89.125


In [8]:
views_diff = pd.read_sql_query("""
SELECT
    diff.uid,
    diff.avg_diff,
    pv.pageviews
FROM (
    SELECT
        t.uid,
        AVG(
            CAST((unixepoch(t.first_commit_ts) - d.deadlines) / 3600 AS INTEGER)
        ) AS avg_diff
    FROM test t
    JOIN deadlines d
      ON t.labname = d.labs
    WHERE t.labname <> 'project1'
    GROUP BY t.uid
) diff
JOIN (
    SELECT
        uid,
        COUNT(*) AS pageviews
    FROM pageviews
    WHERE uid IN (SELECT uid FROM test)
    GROUP BY uid
) pv
ON diff.uid = pv.uid
""", con)

views_diff[['avg_diff', 'pageviews']].corr()

,avg_diff,pageviews
avg_diff,1.000000,-0.279736
pageviews,-0.279736,1.000000
